In [ ]:
import pandas as pd


# 1. Load data (Ensure files are uploaded to the folder icon on the left!)
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
category_name = pd.read_csv('product_category_name_translation.csv')

# 2. Merge systematically using 'left' to keep all orders
df = pd.merge(orders, items, on='order_id', how='left')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, customers, on='customer_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')
df = pd.merge(df, payments, on='order_id', how='left')
df = pd.merge(df, reviews, on='order_id', how='left')
df = pd.merge(df, category_name, on='product_category_name', how='left')

# 3. Check the result
print(f"Final dataset has {df.shape[0]} rows and {df.shape[1]} columns.")
print(df.head())
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
product_category_name = pd.read_csv('product_category_name_translation.csv')

df = pd.merge(orders, items, on='order_id', how='left')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, customers, on='customer_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')
df = pd.merge(df, order_payments, on='order_id', how='left')
df = pd.merge(df, order_reviews, on='order_id', how='left')
df = pd.merge(df, product_category_name, on='product_category_name', how='left')
print(df.head())

# Group by zip code to get the average lat/lon before merging
geo_unique = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

# Now it is safe to merge with your main 'df'
df = pd.merge(df, geo_unique, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

In [ ]:
# 1. Handle Missing Values
# Remove rows where delivery date is missing (likely canceled or pending)
df = df.dropna(subset=['order_delivered_customer_date'])

# Fill missing product category names with 'unknown'
df['product_category_name'] = df['product_category_name'].fillna('unknown')

# Fill missing freight_value with 0
df['freight_value'] = df['freight_value'].fillna(0)


# 2. Filter Data
# Keep only 'delivered' orders
df = df[df['order_status'] == 'delivered']

# Remove orders with negative prices or freight
df = df[(df['price'] >= 0) & (df['freight_value'] >= 0)]

# Filter for the specific date range (2017-01-01 to 2018-08-31)
# We convert the column to datetime first to make filtering possible
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df = df[(df['order_purchase_timestamp'] >= '2017-01-01') &
        (df['order_purchase_timestamp'] <= '2018-08-31')]

# Remove duplicate order_id entries to ensure unique transactions
df = df.drop_duplicates(subset=['order_id'])


# 3. Data Type Conversion
# Convert all date columns to datetime objects
date_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_columns:
    df[col] = pd.to_datetime(df[col])

# Ensure IDs are strings and prices are floats
df['order_id'] = df['order_id'].astype(str)
df['customer_id'] = df['customer_id'].astype(str)
df['product_id'] = df['product_id'].astype(str)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print("Data Cleaning Complete!")
print(f"Remaining rows: {len(df)}")

In [ ]:
# 1. Total number of orders
total_orders = df['order_id'].nunique()

# 2. Total revenue (sum of all prices)
total_revenue = df['price'].sum()

# 3. Average Order Value (AOV)
aov = total_revenue / total_orders

# 4. Average items per order
# We count total rows (items) divided by unique order IDs
avg_items_per_order = len(df) / total_orders

# 5. Date range of data
date_min = df['order_purchase_timestamp'].min()
date_max = df['order_purchase_timestamp'].max()

print(f"Total Orders: {total_orders}")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"AOV: ${aov:.2f}")
print(f"Date Range: {date_min} to {date_max}")

In [ ]:
# Create a Year-Month column for grouping
df['month_year'] = df['order_purchase_timestamp'].dt.to_period('M')

# Grouping by month
monthly_trends = df.groupby('month_year').agg({
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

# Renaming columns for clarity
monthly_trends.columns = ['Month', 'Number_of_Orders', 'Total_Revenue']
monthly_trends['Average_Order_Value'] = monthly_trends['Total_Revenue'] / monthly_trends['Number_of_Orders']

# Calculate Month-over-Month Growth Rate
# Formula: ((Current - Previous) / Previous) * 100
monthly_trends['Growth_Rate'] = monthly_trends['Number_of_Orders'].pct_change() * 100

print(monthly_trends)

In [ ]:
# Extract Day of Week and Hour
df['day_of_week'] = df['order_purchase_timestamp'].dt.day_name()
df['hour'] = df['order_purchase_timestamp'].dt.hour

# Daily Pattern
daily_pattern = df.groupby('day_of_week')['order_id'].nunique().sort_values(ascending=False)
print("Orders by Day of Week:")
print(daily_pattern)

# Hourly Pattern
hourly_pattern = df.groupby('hour')['order_id'].nunique()
peak_hour = hourly_pattern.idxmax()

print(f"\nPeak Ordering Hour: {peak_hour}:00")

In [ ]:
# 1. Create the Category Summary Table
category_analysis = df.groupby('product_category_name_english').agg({
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

# 2. Calculate Required Metrics
# Rename columns for clarity
category_analysis.columns = ['Category', 'Number_of_Orders', 'Total_Revenue']

# Calculate Average Price per category
category_analysis['Average_Price'] = category_analysis['Total_Revenue'] / category_analysis['Number_of_Orders']

# Calculate Percentage of Total Orders
total_orders_count = category_analysis['Number_of_Orders'].sum()
category_analysis['Percentage_of_Total_Orders'] = (category_analysis['Number_of_Orders'] / total_orders_count) * 100

# 3. Identify Top 10 Categories by Volume
top_10_volume = category_analysis.sort_values(by='Number_of_Orders', ascending=False).head(10)

# 4. Identify Top 10 Categories by Revenue
top_10_revenue = category_analysis.sort_values(by='Total_Revenue', ascending=False).head(10)

# Display Results
print("--- Top 10 Categories by Volume ---")
print(top_10_volume[['Category', 'Number_of_Orders', 'Percentage_of_Total_Orders']])

print("\n--- Top 10 Categories by Revenue ---")
print(top_10_revenue[['Category', 'Total_Revenue', 'Average_Price']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting Top 10 by Volume
plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_volume, x='Number_of_Orders', y='Category', palette='viridis')
plt.title('Top 10 Product Categories by Order Volume')
plt.show()

In [ ]:
# Create State Summary
state_analysis = df.groupby('customer_state').agg({
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

# Rename and Calculate Metrics
state_analysis.columns = ['State', 'Number_of_Orders', 'Total_Revenue']
state_analysis['Percentage_of_Total_Orders'] = (state_analysis['Number_of_Orders'] / state_analysis['Number_of_Orders'].sum()) * 100
state_analysis['Average_Order_Value'] = state_analysis['Total_Revenue'] / state_analysis['Number_of_Orders']

# Identify Top and Bottom 5
top_5_states = state_analysis.sort_values(by='Number_of_Orders', ascending=False).head(5)
bottom_5_states = state_analysis.sort_values(by='Number_of_Orders', ascending=True).head(5)

print("--- Top 5 States by Volume ---")
print(top_5_states)

In [ ]:
# Seller State Summary
seller_analysis = df.groupby('seller_state').agg({
    'seller_id': 'nunique',
    'order_id': 'nunique'
}).reset_index()

seller_analysis.columns = ['Seller_State', 'Number_of_Sellers', 'Orders_Fulfilled']
seller_analysis['Avg_Orders_per_Seller'] = seller_analysis['Orders_Fulfilled'] / seller_analysis['Number_of_Sellers']

print("\n--- Seller Distribution ---")
print(seller_analysis.sort_values(by='Number_of_Sellers', ascending=False).head(5))

In [ ]:
# 1. Classify orders
df['delivery_type'] = df.apply(lambda x: 'in-state' if x['customer_state'] == x['seller_state'] else 'cross-state', axis=1)

# 2. Calculate Percentage of Cross-State Orders
cross_state_pct = (df['delivery_type'] == 'cross-state').mean() * 100

# 3. Calculate Delivery Time (in days)
# We subtract purchase date from delivered date
df['delivery_time_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# 4. Compare Delivery Times
delivery_comparison = df.groupby('delivery_type')['delivery_time_days'].mean()

print(f"\nPercentage of Cross-State Orders: {cross_state_pct:.2f}%")
print("\nAverage Delivery Time (Days):")
print(delivery_comparison)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extract the month (1-12) from the timestamp
df['month_num'] = df['order_purchase_timestamp'].dt.month

# 2. Calculate average order volume per month across all years
seasonality = df.groupby('month_num')['order_id'].nunique().reset_index()
seasonality.columns = ['Month', 'Total_Orders']

# 3. Create the Line Chart
plt.figure(figsize=(10, 5))
sns.lineplot(data=seasonality, x='Month', y='Total_Orders', marker='o', color='teal')
plt.title('Monthly Seasonality (Aggregated 2017-2018)')
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Identify the peak
peak_month = seasonality.loc[seasonality['Total_Orders'].idxmax(), 'Month']
print(f"The seasonal peak occurs in Month: {peak_month}")

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# 1. Prepare data: Monthly order counts
trend_data = df.groupby('month_year')['order_id'].nunique().reset_index()
trend_data['Month_Index'] = np.arange(len(trend_data)) # Create numeric x-axis (0, 1, 2...)

# 2. Fit a Linear Trend Line
X = trend_data[['Month_Index']]
y = trend_data['order_id']
model = LinearRegression().fit(X, y)
trend_line = model.predict(X)

# 3. Plot the Actual vs. Trend
plt.figure(figsize=(12, 6))
plt.plot(trend_data['Month_Index'], y, label='Actual Orders', marker='s')
plt.plot(trend_data['Month_Index'], trend_line, label='Linear Trend', linestyle='--', color='red')
plt.title('Order Trend Analysis (2017-01 to 2018-08)')
plt.xlabel('Months since Jan 2017')
plt.ylabel('Number of Orders')
plt.legend()
plt.show()

# Calculate the slope to see if growing
slope = model.coef_[0]
print(f"Trend Slope: {slope:.2f} orders per month.")
if slope > 0:
    print("Conclusion: Orders are increasing over time.")
else:
    print("Conclusion: Orders are decreasing over time.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the visual style for all plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Convert 'Month' column to string for plotting purposes
monthly_trends['Month_str'] = monthly_trends['Month'].astype(str)
seasonality['Month_str'] = seasonality['Month'].astype(str)

# 1. Monthly Order Trend (Line Chart)
plt.figure()
sns.lineplot(data=monthly_trends, x='Month_str', y='Number_of_Orders', marker='o', color='b')
plt.title('Monthly Order Trend (2017-2018)', fontsize=15)
plt.xlabel('Month') # Add x-label for clarity after conversion
plt.xticks(rotation=45)
plt.show()

# 2. Revenue Trend (Line Chart)
plt.figure()
sns.lineplot(data=monthly_trends, x='Month_str', y='Total_Revenue', marker='s', color='g')
plt.title('Monthly Revenue Trend', fontsize=15)
plt.xlabel('Month') # Add x-label for clarity after conversion
plt.xticks(rotation=45)
plt.show()

# 3. Day of Week Pattern (Bar Chart)
# Ensure days are in correct order
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
plt.figure()
sns.barplot(x=daily_pattern.index, y=daily_pattern.values, order=day_order, palette='muted')
plt.title('Orders by Day of the Week', fontsize=15)
plt.ylabel('Number of Orders')
plt.show()

# 4. Top 10 Categories (Horizontal Bar Chart)
plt.figure()
sns.barplot(data=top_10_volume, x='Number_of_Orders', y='Category', palette='viridis')
plt.title('Top 10 Product Categories by Volume', fontsize=15)
plt.show()

# 5. Geographic Orders by State (Bar Chart / Heatmap)
plt.figure()
state_sorted = state_analysis.sort_values('Number_of_Orders', ascending=False)
sns.barplot(data=state_sorted, x='State', y='Number_of_Orders', palette='magma')
plt.title('Total Orders by State', fontsize=15)
plt.show()

# 6. Seasonality Chart (Month-wise Average)
plt.figure()
sns.lineplot(data=seasonality, x='Month_str', y='Total_Orders', marker='o', color='orange', linewidth=2.5)
plt.title('Seasonality: Average Orders per Month', fontsize=15)
plt.xlabel('Month') # Add x-label for clarity after conversion
plt.xticks(rotation=45) # Rotate for better visibility
plt.show()

Milestone-2

In [ ]:
import pandas as pd

# 1. Ensure date columns are in datetime format
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# 2. Calculate Delivery Delay (in days)
# Formula: Actual Delivery Date - Estimated Delivery Date
df['delivery_delay'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days

# 3. Calculate Delivery Time (in days)
# Formula: Actual Delivery Date - Order Purchase Date
df['delivery_time'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# 4. Create Performance Metrics Summary
total_orders = len(df)

avg_delay = df['delivery_delay'].mean()
late_pct = (df['delivery_delay'] > 0).sum() / total_orders * 100
ontime_pct = (df['delivery_delay'] == 0).sum() / total_orders * 100
early_pct = (df['delivery_delay'] < 0).sum() / total_orders * 100
avg_total_time = df['delivery_time'].mean()

# Display the results
print("--- Delivery Performance Metrics ---")
print(f"Average Delivery Delay: {avg_delay:.2f} days")
print(f"Percentage of Late Deliveries: {late_pct:.2f}%")
print(f"Percentage of On-Time Deliveries: {ontime_pct:.2f}%")
print(f"Percentage of Early Deliveries: {early_pct:.2f}%")
print(f"Average Total Delivery Time: {avg_total_time:.2f} days")

Performance by Customer and Seller State

In [ ]:
# Function to create a performance summary table
def get_segment_performance(df, group_col):
    perf = df.groupby(group_col).agg({
        'delivery_delay': 'mean',
        'delivery_time': 'mean',
        'order_id': 'nunique'
    }).reset_index()

    # Calculate % late deliveries for the segment
    late_counts = df[df['delivery_delay'] > 0].groupby(group_col)['order_id'].nunique()
    perf = perf.merge(late_counts, on=group_col, how='left').rename(columns={'order_id_y': 'late_orders', 'order_id_x': 'total_orders'})
    perf['late_orders'] = perf['late_orders'].fillna(0)
    perf['%_late_deliveries'] = (perf['late_orders'] / perf['total_orders']) * 100

    return perf.sort_values('%_late_deliveries', ascending=False)

# Analysis by Customer State
cust_state_perf = get_segment_performance(df, 'customer_state')
print("--- Top 5 Problem Regions (Customer States) ---")
print(cust_state_perf.head(5))

# Analysis by Seller State
seller_state_perf = get_segment_performance(df, 'seller_state')

Performance by Shipping Distance & Order Value

In [ ]:
# A. Shipping Distance Classification
def classify_distance(row):
    if row['customer_city'] == row['seller_city']:
        return 'Same City (< 50 km)'
    elif row['customer_state'] == row['seller_state']:
        return 'Same State (50-500 km)'
    else:
        return 'Cross-state (> 500 km)'

df['distance_category'] = df.apply(classify_distance, axis=1)

# B. Order Value Classification
def classify_value(price):
    if price < 50: return 'Low Value (< R$ 50)'
    elif price <= 200: return 'Medium Value (R$ 50-200)'
    else: return 'High Value (> R$ 200)'

df['value_category'] = df['price'].apply(classify_value)

# C. Calculate Performance for these Categories
dist_perf = get_segment_performance(df, 'distance_category')
value_perf = get_segment_performance(df, 'value_category')

print("\n--- Performance by Shipping Distance ---")
print(dist_perf[['distance_category', 'delivery_time', '%_late_deliveries']])

print("\n--- Performance by Order Value ---")
print(value_perf[['value_category', 'delivery_time', '%_late_deliveries']])

Performance by Product Category

In [ ]:
# Analysis for Top 10 Product Categories
top_10_cats = df['product_category_name_english'].value_counts().head(10).index
cat_perf = get_segment_performance(df[df['product_category_name_english'].isin(top_10_cats)], 'product_category_name_english')

print("\n--- Top 10 Categories Delivery Performance ---")
print(cat_perf[['product_category_name_english', 'delivery_time', '%_late_deliveries']])

Step 3: Identify Delivery Bottlenecks

Top 10 Problem Routes

In [ ]:
# 1. Group by Customer and Seller State
route_bottlenecks = df.groupby(['customer_state', 'seller_state']).agg({
    'order_id': 'nunique',
    'delivery_delay': 'mean'
}).reset_index()

# 2. Identify Late Orders per route to calculate % Late
# We create a temporary indicator for late orders
df['is_late'] = df['delivery_delay'] > 0
late_pct_route = df.groupby(['customer_state', 'seller_state'])['is_late'].mean() * 100

# 3. Merge the % Late back into our bottleneck table
route_bottlenecks = route_bottlenecks.merge(late_pct_route, on=['customer_state', 'seller_state'])

# 4. Rename columns for the required output format
route_bottlenecks.columns = ['Customer State', 'Seller State', 'Orders', 'Avg Delay', '% Late']

# 5. Filter for statistical relevance (Minimum 50 orders)
route_bottlenecks = route_bottlenecks[route_bottlenecks['Orders'] >= 50]

# 6. Sort to find the worst performers
top_10_bottlenecks = route_bottlenecks.sort_values(by='% Late', ascending=False).head(10)

print("--- Top 10 Problem Routes (Min. 50 Orders) ---")
print(top_10_bottlenecks)

Seasonal Impact on Delivery

In [ ]:
# 1. Group by Month (using the 'month_year' column we created in Milestone 1)
# Re-create 'month_year' as the df might have been overwritten and lost this column
df['month_year'] = df['order_purchase_timestamp'].dt.to_period('M')

seasonal_delivery = df.groupby('month_year').agg({
    'delivery_delay': 'mean',
    'is_late': 'mean'
}).reset_index()

seasonal_delivery['%_Late'] = seasonal_delivery['is_late'] * 100

# 2. Find the worst month
worst_month = seasonal_delivery.loc[seasonal_delivery['delivery_delay'].idxmax()]

print("\n--- Seasonal Delivery Impact ---")
print(f"The month with the worst delivery performance was: {worst_month['month_year']}")
print(f"Average Delay that month: {worst_month['delivery_delay']:.2f} days")
print(f"Percentage of Late Orders: {worst_month['%_Late']:.2f}%")

# Step 4: Demand Forecasting - Data Preparation

Select Top Categories for Forecasting

In [ ]:
# 1. Identify top 5 categories
top_forecast_cats = df['product_category_name_english'].value_counts().head(5).index.tolist()
print(f"Categories selected for forecasting: {top_forecast_cats}")

# 2. Filter the main dataframe to include only these categories
forecast_df = df[df['product_category_name_english'].isin(top_forecast_cats)].copy()

Create Daily Time Series Data

In [ ]:
# Function to prepare daily data for a specific category
def prepare_category_ts(data, category_name):
    # Filter for the specific category
    cat_data = data[data['product_category_name_english'] == category_name]

    # Aggregate by date
    ts = cat_data.groupby(cat_data['order_purchase_timestamp'].dt.date).agg({
        'order_id': 'nunique'
    }).rename(columns={'order_id': 'demand'})

    # Ensure the index is a DatetimeIndex
    ts.index = pd.to_datetime(ts.index)

    # Fill Missing Dates: Create a complete range from start to end
    all_dates = pd.date_range(start=ts.index.min(), end=ts.index.max(), freq='D')
    ts = ts.reindex(all_dates, fill_value=0)

    return ts

# Example: Prepare the series for the #1 category
primary_cat = top_forecast_cats[0]
ts_data = prepare_category_ts(forecast_df, primary_cat)

print(f"\n--- Daily Demand Data for {primary_cat} ---")
print(ts_data.head())

Split Data into Training and Test Sets

In [ ]:
# Calculate the split point
split_idx = int(len(ts_data) * 0.8)

train_data = ts_data.iloc[:split_idx]
test_data = ts_data.iloc[split_idx:]

print(f"\nTraining set size: {len(train_data)} days")
print(f"Test set size: {len(test_data)} days")
print(f"Forecast period starts on: {test_data.index.min()}")

# Step 5: Time Series Forecasting

Method 1: Moving Average (MA)

In [ ]:
# Assuming 'ts_data' is the daily demand for your top category from Step 4
forecast_results = test_data.copy()

# 1. Calculate Moving Averages on the training data to initialize
# We use the full series to ensure the 'test' period has rolling values
ts_data['MA_7'] = ts_data['demand'].rolling(window=7).mean()
ts_data['MA_30'] = ts_data['demand'].rolling(window=30).mean()

# 2. Extract the forecast for the test period
forecast_results['MA_7_Forecast'] = ts_data['MA_7'].iloc[split_idx:]
forecast_results['MA_30_Forecast'] = ts_data['MA_30'].iloc[split_idx:]

print("--- Moving Average Forecast (First 5 Days) ---")
print(forecast_results[['demand', 'MA_7_Forecast']].head())

Method 2: Exponential Smoothing

In [ ]:
def simple_exponential_smoothing(series, alpha):
    result = [series[0]] # First forecast is just the first actual value
    for n in range(1, len(series)):
        result.append(alpha * series[n-1] + (1 - alpha) * result[n-1])
    return result

# 1. Run for different Alpha values as requested (0.1, 0.3, 0.5)
forecast_results['SES_0.1'] = simple_exponential_smoothing(ts_data['demand'].values, 0.1)[split_idx:]
forecast_results['SES_0.3'] = simple_exponential_smoothing(ts_data['demand'].values, 0.3)[split_idx:]
forecast_results['SES_0.5'] = simple_exponential_smoothing(ts_data['demand'].values, 0.5)[split_idx:]

print("\n--- Exponential Smoothing Forecast (Alpha=0.3) ---")
print(forecast_results[['demand', 'SES_0.3']].head())

Visualizing the Forecast (Not given in Project)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 7))
plt.plot(test_data.index, test_data['demand'], label='Actual Demand', color='black', alpha=0.3)
plt.plot(test_data.index, forecast_results['MA_7_Forecast'], label='7-Day MA', linestyle='--')
plt.plot(test_data.index, forecast_results['SES_0.3'], label='SES (Alpha=0.3)', color='red')

plt.title(f'Demand Forecast vs Actuals: {primary_cat}')
plt.legend()
plt.show()

# Step 6: Forecast Evaluation

In [ ]:
import numpy as np

def calculate_metrics(actual, forecast):
    # Filter out zeros to avoid division by zero in MAPE
    mask = actual > 0
    a = actual[mask]
    f = forecast[mask]

    mae = np.mean(np.abs(a - f))
    mape = np.mean(np.abs((a - f) / a)) * 100
    rmse = np.sqrt(np.mean((a - f)**2))

    return mae, mape, rmse

# 1. Evaluate Moving Average (MA_7)
mae_ma, mape_ma, rmse_ma = calculate_metrics(test_data['demand'], forecast_results['MA_7_Forecast'])

# 2. Evaluate Exponential Smoothing (SES_0.3)
mae_ses, mape_ses, rmse_ses = calculate_metrics(test_data['demand'], forecast_results['SES_0.3'])

# 3. Create Comparison Table
evaluation_df = pd.DataFrame({
    'Category': [primary_cat, primary_cat],
    'Method': ['MA_7', 'Exp. Smoothing (0.3)'],
    'MAE': [mae_ma, mae_ses],
    'MAPE (%)': [mape_ma, mape_ses],
    'RMSE': [rmse_ma, rmse_ses]
})

# Identify the best model based on lowest MAPE
evaluation_df['Best?'] = evaluation_df['MAPE (%)'] == evaluation_df['MAPE (%)'].min()

print("--- Forecast Evaluation Table ---")
print(evaluation_df)

# Step 7: Generate Future Forecasts

In [ ]:
# 1. Select your best model (Assuming SES with Alpha 0.3 was best)
# We use the last available forecast value as the starting point for the future
future_days = 30
last_date = ts_data.index.max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=future_days)

# For Simple Exponential Smoothing, the 'future' forecast remains constant
# at the last calculated level (as it assumes the most recent level persists)
last_forecast_value = forecast_results['SES_0.3'].iloc[-1]
future_forecast = [last_forecast_value] * future_days

# 2. Create the Future Forecast Table
forecast_30_days = pd.DataFrame({
    'Date': future_dates,
    'Forecasted_Demand': future_forecast
})

print("--- 30-Day Future Demand Forecast ---")
print(forecast_30_days.head(10))

# 3. Plot Actual vs. Forecasted Values
plt.figure(figsize=(15, 6))
# Plot historical data (last 60 days for clarity)
plt.plot(ts_data.index[-60:], ts_data['demand'].iloc[-60:], label='Historical Demand', color='blue')
# Plot future forecast
plt.plot(future_dates, future_forecast, label='30-Day Future Projection', color='red', linestyle='--')

plt.title(f'Future 30-Day Demand Forecast: {primary_cat}', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Number of Orders')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Step 8: Create Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set overall aesthetic style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 1. Delivery Performance Map (Bar chart as proxy for Heatmap)
# Showing % late deliveries by state
plt.figure()
state_perf_sorted = cust_state_perf.sort_values('%_late_deliveries', ascending=False)
sns.barplot(data=state_perf_sorted, x='customer_state', y='%_late_deliveries', palette='Reds_r')
plt.title('Percentage of Late Deliveries by Customer State', fontsize=14)
plt.ylabel('% Late Deliveries')
plt.show()

# 2. Delay Distribution (Histogram)
# Showing how many days orders are typically delayed
plt.figure()
# We filter for only late orders to see the distribution of actual delays
sns.histplot(df[df['delivery_delay'] > 0]['delivery_delay'], bins=30, kde=True, color='orange')
plt.title('Distribution of Delivery Delays (Late Orders Only)', fontsize=14)
plt.xlabel('Days Delayed')
plt.show()

# 3. Performance by Category (Bar Chart)
# Comparing average delivery times across top categories
plt.figure()
sns.barplot(data=cat_perf, x='delivery_time', y='product_category_name_english', palette='viridis')
plt.title('Average Delivery Time by Product Category', fontsize=14)
plt.xlabel('Average Days to Deliver')
plt.show()

# 4. Time Series (Line Chart)
# Showing actual demand over time for the top category
plt.figure()
plt.plot(ts_data.index, ts_data['demand'], color='blue', label='Daily Demand')
plt.title(f'Actual Demand Over Time: {primary_cat}', fontsize=14)
plt.legend()
plt.show()

# 5. Forecast Accuracy (Line Chart)
# Comparing actual vs forecast for the test period
plt.figure()
plt.plot(test_data.index, test_data['demand'], label='Actual', color='black', alpha=0.5)
plt.plot(test_data.index, forecast_results['SES_0.3'], label='Forecast (SES 0.3)', color='red', linestyle='--')
plt.title('Forecast Accuracy: Actual vs. Predicted (Test Period)', fontsize=14)
plt.legend()
plt.show()

# 6. Forecast Chart (Historical + 30-Day Forecast)
# Showing the complete picture including the future
plt.figure()
plt.plot(ts_data.index[-90:], ts_data['demand'].iloc[-90:], label='Historical (Last 90 Days)')
plt.plot(forecast_30_days['Date'], forecast_30_days['Forecasted_Demand'],
         label='30-Day Future Forecast', color='green', linewidth=2)
plt.title('Final Demand Forecast: Next 30 Days', fontsize=14)
plt.fill_between(forecast_30_days['Date'], forecast_30_days['Forecasted_Demand']*0.9,
                 forecast_30_days['Forecasted_Demand']*1.1, color='green', alpha=0.1, label='Confidence Interval')
plt.legend()
plt.show()

# **Milestone 3**

Step 1: Product Selection for Inventory Analysis

In [ ]:
# 1. Filter for the most relevant time period (to ensure consistent demand)
# We will focus on products with sales spanning at least 6 months
product_lifespan = df.groupby('product_id')['order_purchase_timestamp'].agg(['min', 'max'])
product_lifespan['duration_days'] = (product_lifespan['max'] - product_lifespan['min']).dt.days
stable_products = product_lifespan[product_lifespan['duration_days'] > 180].index

# 2. Aggregate sales metrics by Product ID
product_master = df[df['product_id'].isin(stable_products)].groupby('product_id').agg({
    'product_category_name_english': 'first',
    'order_item_id': 'count',           # Total units sold
    'price': 'sum',                     # Total revenue
    'order_purchase_timestamp': 'nunique' # Number of unique days with sales
}).reset_index()

# 3. Calculate Average Daily Demand and Standard Deviation
# We create a daily demand series for each product to get its volatility
daily_demand = df.groupby(['product_id', df['order_purchase_timestamp'].dt.date])['order_item_id'].count().reset_index()

avg_demand = daily_demand.groupby('product_id')['order_item_id'].mean()
std_demand = daily_demand.groupby('product_id')['order_item_id'].std()

# 4. Merge stats back to the master list
product_master = product_master.merge(avg_demand.rename('avg_daily_demand'), on='product_id')
product_master = product_master.merge(std_demand.rename('std_dev_demand'), on='product_id')

# 5. Select the final 15-20 products
# We sort by high sales volume and select top performers from different categories
final_selection = product_master.sort_values(by='order_item_id', ascending=False).head(20)

# 6. Formatting columns as per the requirement
final_selection.columns = [
    'Product ID', 'Product Category', 'Total Units Sold',
    'Total Revenue', 'Days with Sales', 'Average Daily Demand', 'Std Deviation Demand'
]

print("--- Product Master List (Top 20 Products) ---")
print(final_selection[['Product ID', 'Product Category', 'Total Units Sold', 'Average Daily Demand', 'Std Deviation Demand']].head(10))

Step 2: Calculate Inventory Parameters

In [ ]:
import numpy as np

# 1. Constants and Assumptions (from your instructions)
S = 100            # Ordering Cost (R$ per order)
H_rate = 0.20      # Holding Cost Rate (20% per year)
L = 7              # Lead Time (7 days)

# 2. Calculate time duration of the dataset in days
total_days = (df['order_purchase_timestamp'].max() - df['order_purchase_timestamp'].min()).days

# 3. Create a new dataframe for Inventory Parameters
inventory_params = final_selection.copy()

# 4. Calculate Annual Demand (D)
# Formula: (Total units in dataset / total days) * 365
inventory_params['Annual Demand (D)'] = (inventory_params['Total Units Sold'] / total_days) * 365

# 5. Unit Cost (C)
# Formula: Total Revenue / Total Units Sold
inventory_params['Unit Cost (C)'] = inventory_params['Total Revenue'] / inventory_params['Total Units Sold']

# 6. Calculate Holding Cost (H)
# Formula: Unit Cost * Holding Cost Rate
inventory_params['Holding Cost (H)'] = inventory_params['Unit Cost (C)'] * H_rate

# 7. Calculate Sigma (Standard Deviation of demand during lead time)
# Note: For inventory models, we need the standard deviation relative to the lead time period
inventory_params['Lead Time Std Dev (sigma_L)'] = inventory_params['Std Deviation Demand'] * np.sqrt(L)

# 8. Display the parameter table
cols_to_show = ['Product ID', 'Annual Demand (D)', 'Unit Cost (C)', 'Holding Cost (H)', 'Lead Time Std Dev (sigma_L)']
print("--- Inventory Parameters for EOQ Modeling ---")
print(inventory_params[cols_to_show].head())

Step 3: Economic Order Quantity (EOQ)

In [ ]:
import numpy as np

# 1. Calculate Economic Order Quantity (EOQ)
# Formula: sqrt( (2 * D * S) / H )
inventory_params['EOQ'] = np.sqrt(
    (2 * inventory_params['Annual Demand (D)'] * S) / inventory_params['Holding Cost (H)']
).round(0)

# 2. Calculate Associated Annual Costs
# N = Number of orders per year
inventory_params['Orders per Year (N)'] = (
    inventory_params['Annual Demand (D)'] / inventory_params['EOQ']
).round(1)

# Annual Ordering Cost = N * S
inventory_params['Annual Ordering Cost'] = (
    inventory_params['Orders per Year (N)'] * S
)

# Annual Holding Cost = (EOQ / 2) * H
inventory_params['Annual Holding Cost'] = (
    (inventory_params['EOQ'] / 2) * inventory_params['Holding Cost (H)']
)

# Total Annual Inventory Cost
inventory_params['Total Annual Cost'] = (
    inventory_params['Annual Ordering Cost'] + inventory_params['Annual Holding Cost']
)

# 3. Display the Results Table
cols_to_display = [
    'Product ID', 'EOQ', 'Orders per Year (N)',
    'Annual Ordering Cost', 'Annual Holding Cost', 'Total Annual Cost'
]

print("--- Economic Order Quantity (EOQ) Summary ---")
print(inventory_params[cols_to_display].head(10))

Step 4: Reorder Point & Safety Stock

In [ ]:
# 1. Constants for Safety Stock
Z = 1.65  # Service level factor for 95% protection against stockouts
L = 7     # Lead time in days

# 2. Calculate Safety Stock
# Formula: Z * sigma_daily * sqrt(L)
# Note: We use the daily standard deviation calculated in Step 1
inventory_params['Safety Stock'] = (
    Z * inventory_params['Std Deviation Demand'] * np.sqrt(L)
).round(0)

# 3. Calculate Reorder Point (ROP)
# Formula: (Average Daily Demand * Lead Time) + Safety Stock
inventory_params['ROP'] = (
    (inventory_params['Average Daily Demand'] * L) + inventory_params['Safety Stock']
).round(0)

# 4. Calculate Maximum and Average Inventory Levels
# Maximum Inventory = ROP + EOQ
inventory_params['Max Inventory'] = (
    inventory_params['ROP'] + inventory_params['EOQ']
)

# Average Inventory = (EOQ / 2) + Safety Stock
inventory_params['Avg Inventory'] = (
    (inventory_params['EOQ'] / 2) + inventory_params['Safety Stock']
).round(0)

# 5. Create the Final Inventory Policy Table
inventory_policy = inventory_params[[
    'Product ID', 'EOQ', 'Safety Stock', 'ROP', 'Max Inventory', 'Avg Inventory'
]]

print("--- Inventory Policy Table (Replenishment Strategy) ---")
print(inventory_policy.head(10))

Step 5: ABC Analysis

In [ ]:
# 1. Calculate Revenue Contribution for every product
abc_analysis = df.groupby('product_id')['payment_value'].sum().reset_index()
abc_analysis = abc_analysis.sort_values(by='payment_value', ascending=False)

# 2. Calculate Cumulative Revenue and Percentage
total_revenue = abc_analysis['payment_value'].sum()
abc_analysis['revenue_contribution'] = (abc_analysis['payment_value'] / total_revenue) * 100
abc_analysis['cumulative_revenue_pct'] = abc_analysis['revenue_contribution'].cumsum()

# 3. Classify into A, B, and C
# A: Top 80% Revenue (usually ~20% of products)
# B: Next 15% Revenue (usually ~30% of products)
# C: Bottom 5% Revenue (usually ~50% of products)
def classify_abc(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

abc_analysis['Category'] = abc_analysis['cumulative_revenue_pct'].apply(classify_abc)

# 4. Create ABC Summary Table
abc_summary = abc_analysis.groupby('Category').agg({
    'product_id': 'count',
    'payment_value': 'sum'
}).reset_index()

abc_summary.columns = ['Category', 'Number of Products', 'Total Revenue']
abc_summary['% of Products'] = (abc_summary['Number of Products'] / abc_summary['Number of Products'].sum()) * 100
abc_summary['% of Revenue'] = (abc_summary['Total Revenue'] / total_revenue) * 100

print("--- ABC Classification Summary ---")
print(abc_summary.round(2))

Assign Inventory Policies

In [ ]:
import numpy as np

# 1. Define the Policy Mapping
# We set higher Service Levels (Z-scores) for 'A' items to minimize stockouts
policy_map = {
    'A': {'service_level': 0.99, 'z_score': 2.33, 'review_freq': 'Daily',   'priority': 'Highest'},
    'B': {'service_level': 0.95, 'z_score': 1.65, 'review_freq': 'Weekly',  'priority': 'Medium'},
    'C': {'service_level': 0.90, 'z_score': 1.28, 'review_freq': 'Monthly', 'priority': 'Low'}
}

# 2. Merge ABC classification with the product parameters
# 'abc_analysis' comes from your previous step; 'inventory_params' from Milestone 2
inventory_params = inventory_params.rename(columns={'Product ID': 'product_id'}) # Rename column for consistent merge
final_policy = inventory_params.merge(abc_analysis[['product_id', 'Category']], on='product_id')

# 3. Apply the Policy Logic
final_policy['Service Level'] = final_policy['Category'].map(lambda x: policy_map[x]['service_level'])
final_policy['Z_Score'] = final_policy['Category'].map(lambda x: policy_map[x]['z_score'])
final_policy['Review Frequency'] = final_policy['Category'].map(lambda x: policy_map[x]['review_freq'])
final_policy['Priority'] = final_policy['Category'].map(lambda x: policy_map[x]['priority'])

# 4. Recalculate Safety Stock based on the specific Service Level
# Formula: Z * StdDev_Demand * sqrt(LeadTime)
final_policy['Optimized Safety Stock'] = (
    final_policy['Z_Score'] * final_policy['Std Deviation Demand'] * np.sqrt(7) # Assuming 7-day lead time
).round(0)

# 5. Calculate Final Reorder Point (ROP)
final_policy['Final ROP'] = (
    (final_policy['Average Daily Demand'] * 7) +
    final_policy['Optimized Safety Stock']
).round(0)

# Display the summary
policy_summary = final_policy[[
    'product_id', 'Category', 'Priority', 'Review Frequency',
    'Optimized Safety Stock', 'Final ROP'
]]

print("--- Optimized Inventory Policies by Category ---")
print(policy_summary.head(15))

# Step 6: Inventory Cost Analysis

In [ ]:
# 1. Constants for Current State Estimation (as per instructions)
current_Q = 200  # Assumed random order quantity

# 2. Calculate Costs for Current State
# Current Cost = (D / 200) * S + (200 / 2) * H
final_policy['Current Annual Cost'] = (
    (final_policy['Annual Demand (D)'] / current_Q) * S +
    (current_Q / 2) * final_policy['Holding Cost (H)']
)

# 3. Calculate Costs for Proposed State (from your EOQ + Safety Stock results)
# Proposed Cost = (D / EOQ) * S + (EOQ / 2 + Safety Stock) * H
final_policy['Proposed Annual Cost'] = (
    (final_policy['Annual Demand (D)'] / final_policy['EOQ']) * S +
    (final_policy['EOQ'] / 2 + final_policy['Optimized Safety Stock']) * final_policy['Holding Cost (H)']
)

# 4. Calculate Savings
final_policy['Annual Savings'] = final_policy['Current Annual Cost'] - final_policy['Proposed Annual Cost']
final_policy['Savings %'] = (final_policy['Annual Savings'] / final_policy['Current Annual Cost']) * 100

# 5. Calculate Investment Required (Safety Stock Value)
# Safety Stock Investment = Safety Stock * Unit Cost
final_policy['Safety Stock Investment'] = final_policy['Optimized Safety Stock'] * final_policy['Unit Cost (C)']

# 6. Sum up results for ROI Calculation
total_savings = final_policy['Annual Savings'].sum()
total_investment = final_policy['Safety Stock Investment'].sum()
roi = (total_savings / total_investment) * 100

print(f"--- Financial Impact Summary ---")
print(f"Total Annual Savings: R$ {total_savings:,.2f}")
print(f"Total Investment in Safety Stock: R$ {total_investment:,.2f}")
print(f"Project ROI: {roi:.2f}%")

Step 7: Stockout Risk Analysis

In [ ]:
from scipy.stats import norm

# 1. Define Current State (No Safety Stock)
# In the current state, we assume inventory is exactly the Mean Lead Time Demand
# Mean Lead Time Demand = Avg Daily Demand * Lead Time (7)
final_policy['Mean_LT_Demand'] = final_policy['Average Daily Demand'] * 7
final_policy['Current_Inventory'] = final_policy['Mean_LT_Demand']

# 2. Calculate Current Stockout Risk
# Z-score = (Current Inventory - Mean LT Demand) / (Daily StdDev * sqrt(LT))
# Since Current Inv = Mean LT Demand, Z = 0
final_policy['Current_Z'] = (final_policy['Current_Inventory'] - final_policy['Mean_LT_Demand']) / (final_policy['Std Deviation Demand'] * np.sqrt(7))
final_policy['Current_Stockout_Risk'] = (1 - norm.cdf(final_policy['Current_Z'])) * 100

# 3. Calculate Proposed Stockout Risk (With Safety Stock)
# New Inventory = Mean LT Demand + Safety Stock
final_policy['Proposed_Inventory'] = final_policy['Mean_LT_Demand'] + final_policy['Optimized Safety Stock']
final_policy['Proposed_Z'] = (final_policy['Proposed_Inventory'] - final_policy['Mean_LT_Demand']) / (final_policy['Std Deviation Demand'] * np.sqrt(7))
final_policy['Proposed_Stockout_Risk'] = (1 - norm.cdf(final_policy['Proposed_Z'])) * 100

# 4. Results Table
risk_analysis = final_policy[['product_id', 'Category', 'Current_Stockout_Risk', 'Proposed_Stockout_Risk']]
print("--- Stockout Risk Analysis (%) ---")
print(risk_analysis.head(10))

Step 8: Create Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set overall aesthetic style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 1. ABC Curve (Pareto Chart)
plt.figure()
# Plotting cumulative revenue %
plt.plot(range(len(abc_analysis)), abc_analysis['cumulative_revenue_pct'], color='blue', linewidth=2, label='Cumulative Revenue %')
# Vertical markers for A, B, and C cut-offs
plt.axhline(y=80, color='r', linestyle='--', label='A-Items (80% Revenue)')
plt.axhline(y=95, color='orange', linestyle='--', label='B-Items (95% Revenue)')
plt.fill_between(range(len(abc_analysis)), 0, 80, color='red', alpha=0.1)
plt.title('ABC Analysis: Pareto Revenue Curve', fontsize=14)
plt.xlabel('Number of Products (Ranked by Revenue)')
plt.ylabel('Cumulative Revenue %')
plt.legend()
plt.show()

# 2. EOQ vs Current Order Quantity (Bar Chart)
plt.figure()
# Select a sample of top products for clarity
sample_df = final_policy.head(10)
x = np.arange(len(sample_df))
width = 0.35

plt.bar(x - width/2, [200]*len(sample_df), width, label='Current State (Q=200)', color='gray', alpha=0.6)
plt.bar(x + width/2, sample_df['EOQ'], width, label='Proposed EOQ', color='green')
plt.xticks(x, sample_df['product_id'].str[:8], rotation=45)
plt.title('Optimization: Current Order Quantity vs. Calculated EOQ', fontsize=14)
plt.ylabel('Units per Order')
plt.legend()
plt.show()

# 3. Inventory Levels (Stacked Bar Chart)
# Showing ROP, Safety Stock, and EOQ for Top 10 Products
plt.figure()
# We stack ROP on top of Safety Stock, then EOQ
plt.bar(sample_df['product_id'].str[:8], sample_df['Safety Stock'], label='Safety Stock', color='red')
plt.bar(sample_df['product_id'].str[:8], sample_df['ROP'], bottom=sample_df['Safety Stock'], label='Reorder Point (ROP)', color='orange')
plt.bar(sample_df['product_id'].str[:8], sample_df['EOQ'], bottom=sample_df['ROP'], label='EOQ', color='green')
plt.title('Inventory Strategy: Safety Stock, ROP, and EOQ Distribution', fontsize=14)
plt.xticks(rotation=45)
plt.ylabel('Units')
plt.legend()
plt.show()

# 4. Cost Savings (Bar Chart)
plt.figure()
sns.barplot(data=sample_df, x='product_id', y='Annual Savings', palette='viridis')
plt.title('Projected Annual Savings per Product', fontsize=14)
plt.xticks(ticks=range(10), labels=sample_df['product_id'].str[:8], rotation=45)
plt.ylabel('Annual Savings (R$)')
plt.show()

# 5. Inventory Investment (Pie Chart)
plt.figure()
# Aggregating investment (Safety Stock Value) by ABC category
invest_summary = final_policy.groupby('Category')['Safety Stock Investment'].sum()
plt.pie(invest_summary, labels=invest_summary.index, autopct='%1.1f%%', colors=['#ff9999','#66b3ff','#99ff99'], startangle=140)
plt.title('Capital Tied in Inventory by ABC Category', fontsize=14)
plt.show()

# Milestone **4**

Step 1: KPI Defnition

In [ ]:
import pandas as pd
import numpy as np

# --- 1. Order Fulfillment KPIs ---
# Order Fill Rate: (Orders delivered / Total orders) * 100
total_orders = df['order_id'].nunique()
fulfilled_orders = df[df['order_status'] == 'delivered']['order_id'].nunique()
order_fill_rate = (fulfilled_orders / total_orders) * 100

# Order Cycle Time: Average time from order to delivery
# Using your calculated delivery_time column from Milestone 1
avg_order_cycle_time = df['delivery_time'].mean()

# --- 2. Inventory KPIs ---
# Inventory Turnover Ratio: Total Sales (proxy for COGS) / Avg Inventory Value
# We use the average inventory from your Milestone 3 results
avg_inventory_value = (final_policy['Avg Inventory'] * final_policy['Unit Cost (C)']).sum()
total_sales_value = df['payment_value'].sum()
inventory_turnover = total_sales_value / avg_inventory_value

# Days of Inventory: 365 / Inventory Turnover
days_of_inventory = 365 / inventory_turnover

# --- 3. Delivery KPIs ---
# On-Time Delivery Rate: % of orders delivered by estimated date
# We use the 'is_late' flag from Milestone 1
on_time_delivery_rate = (len(df[df['is_late'] == 0]) / len(df)) * 100

# Delivery Delay: Average days of delay for LATE orders only
avg_delivery_delay = df[df['delivery_delay'] > 0]['delivery_delay'].mean()

# --- 4. Cost & Forecast KPIs ---
# Freight Cost as % of Order Value
total_freight = df['freight_value'].sum()
freight_pct_revenue = (total_freight / total_sales_value) * 100

# Forecast Accuracy: 100% - MAPE (using the test data from Milestone 2)
# Retrieve the MAPE from the best performing model in evaluation_df
mape = evaluation_df[evaluation_df['Best?'] == True]['MAPE (%)'].iloc[0]
forecast_accuracy = 100 - mape

# --- KPI Summary Table ---
kpi_summary = pd.DataFrame({
    'Metric Category': ['Fulfillment', 'Fulfillment', 'Inventory', 'Inventory', 'Delivery', 'Delivery', 'Cost', 'Forecast'],
    'KPI Name': ['Order Fill Rate', 'Order Cycle Time', 'Inventory Turnover', 'Days of Inventory', 'On-Time Delivery Rate', 'Avg Delay (Late)', 'Freight Cost %', 'Forecast Accuracy'],
    'Current Value': [
        f"{order_fill_rate:.2f}%", f"{avg_order_cycle_time:.2f} Days", f"{inventory_turnover:.2f}x",
        f"{days_of_inventory:.1f} Days", f"{on_time_delivery_rate:.2f}%", f"{avg_delivery_delay:.2f} Days",
        f"{freight_pct_revenue:.2f}%", f"{forecast_accuracy:.2f}%"
    ],
    'Target Value': ['>98%', '<10 Days', '>12.0x', '<30 Days', '>90%', '<2 Days', '<10%', '>85%']
})

print(kpi_summary)